# ⚛️ Vigyan-7B: Frontier STEM Reasoning Model (Non-Commercial Demo)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shreyansh001boy-tech/vigyan-7b-demo/blob/main/vigyan_7b_demo.ipynb)
&nbsp;[![License: CC BY-NC 4.0](https://img.shields.io/badge/License-CC_BY--NC_4.0-lightgrey.svg)](https://creativecommons.org/licenses/by-nc/4.0/)
&nbsp;[![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-Vigyan--7B--STEM--DPO-yellow)](https://huggingface.co/shreyansh12183/Vigyan-7B-STEM-DPO-v1)

---

### 🌟 Overview
**Vigyan-7B-STEM-DPO-v1** is India's first specialized open-weight STEM Small Language Model (SLM), fine-tuned and aligned using **Direct Preference Optimization (DPO)** to eliminate arithmetic drift, enforce physical conservation laws, and invoke deterministic execution tools.

> **Hardware Requirement:** Free Google Colab **Tesla T4 GPU** (Allocates in ~5.5 GB VRAM using 4-bit NF4 quantization).
> **License:** Creative Commons Attribution-NonCommercial 4.0 International (**CC BY-NC 4.0**). Free for academic, educational, and research use.

### Step 1: Install Dependencies (Runs in ~30 seconds)

In [ ]:
# Install lightweight runtime dependencies
!pip install -q transformers peft accelerate bitsandbytes gradio
print('✅ Environment ready.')

### Step 2: Load Model in 4-bit Precision (~5.5 GB VRAM)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

BASE_MODEL_ID = 'shreyansh12183/Shreyansh-STEM-AI-7B-Final'
ADAPTER_ID = 'shreyansh12183/Vigyan-7B-STEM-DPO-v1'

print(f'⚡ PyTorch version: {torch.__version__} | CUDA Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'⚡ Device: {torch.cuda.get_device_name(0)}')

print('📥 Loading Tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print('📥 Loading 7B Base Weights in 4-bit NF4...')
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.float16
)

print('📥 Attaching Vigyan-7B DPO LoRA Adapter...')
model = PeftModel.from_pretrained(base_model, ADAPTER_ID)
model.eval()

if hasattr(model, 'generation_config') and hasattr(model.generation_config, 'stop_strings'):
    model.generation_config.stop_strings = None

print('🎉 Vigyan-7B DPO Model Loaded Successfully!')

### Step 3: Launch Interactive Gradio Web UI (with Public Share Link)

In [ ]:
import gradio as gr
import time

def solve_stem_problem(question, max_tokens, temperature):
    if not question.strip():
        return 'Please enter a valid STEM problem.'
    
    prompt = f'<|im_start|>user\n{question.strip()}\nSolve step by step. Show full mathematical reasoning and invoke Python tools where necessary. State the final answer clearly in \\boxed{{number}}.<|im_end|>\n<|im_start|>assistant\n'
    
    inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
    t0 = time.time()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            tokenizer=tokenizer,
            max_new_tokens=int(max_tokens),
            do_sample=True if temperature > 0.05 else False,
            temperature=float(temperature) if temperature > 0.05 else None,
            pad_token_id=tokenizer.eos_token_id
        )
    dt = time.time() - t0
    
    gen_tokens = outputs[0][inputs.input_ids.shape[1]:]
    response = tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()
    
    tokens_count = len(gen_tokens)
    speed = round(tokens_count / dt, 1) if dt > 0 else 0.0
    
    footer = f'\n\n---\n⚡ **Latency:** {dt:.2f}s | **Tokens:** {tokens_count} | **Throughput:** {speed} tok/s'
    return response + footer

examples = [
    ['Two precision resistors R1 = 150 ohms and R2 = 300 ohms are connected in parallel. Calculate the equivalent resistance in ohms.'],
    ['A circuit has Thevenin voltage V_th = 24 V and Thevenin resistance R_th = 6 ohms. What is the maximum power in Watts (W) deliverable to a variable load?'],
    ['A polysilicon interconnect wire has sheet resistance R_sheet = 50 ohms/sq, length L = 200 micrometers, and width W = 0.5 micrometers. Calculate total resistance in ohms.'],
    ['A sounding rocket payload of mass m = 100 kg is lifted to an altitude of h = 50 m. Using standard g = 9.8 m/s^2, calculate its gravitational potential energy in Joules.'],
    ['Evaluate the definite integral: \\int_{0}^{4} (3x^2 + 2x)\,dx. State the exact integer value in \\boxed{}.']
]

with gr.Blocks(theme=gr.themes.Soft(primary_hue='indigo')) as demo:
    gr.Markdown(
        '''
        # ⚛️ Vigyan-7B STEM AI: Interactive Live Reasoning Demo
        ### High-Precision Engineering, Circuit Physics & Mathematics Assistant
        *Vigyan AI — Varanasi, India | Non-Commercial Research Release (CC BY-NC 4.0)*
        '''
    )
    
    with gr.Row():
        with gr.Column(scale=4):
            input_box = gr.Textbox(
                label='Enter STEM Problem / Engineering Prompt',
                placeholder='Type your physics, circuit, VLSI, or calculus question here...',
                lines=4
            )
            with gr.Row():
                max_toks = gr.Slider(minimum=128, maximum=1024, value=512, step=64, label='Max Tokens')
                temp = gr.Slider(minimum=0.0, maximum=1.0, value=0.1, step=0.05, label='Temperature (0.0 = Deterministic)')
            submit_btn = gr.Button('🚀 Compute Solution with Vigyan-7B', variant='primary')
            
            gr.Examples(
                examples=examples,
                inputs=input_box,
                label='Quick One-Click Test Cases'
            )
            
        with gr.Column(scale=5):
            output_box = gr.Markdown(label='Model Derivation & Deterministic Output')
            
    submit_btn.click(
        fn=solve_stem_problem,
        inputs=[input_box, max_toks, temp],
        outputs=output_box
    )

print('🚀 Launching Gradio Web UI with public share link...')
demo.launch(share=True, debug=False)
